---
#**Instalação**
---

In [ ]:
!pip install pyarrow matplotlib seaborn ipywidgets -q


In [ ]:
#import zipfile   # [REMOVIDO] só era usado para abrir os .zip
#import io        # [REMOVIDO] só era usado para ler o conteúdo do .zip
import requests
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm
import json
from sklearn.preprocessing import MinMaxScaler
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from IPython.display import display
import numpy as np

---
#**Configurações**
---

In [ ]:
TABLE_SCR  = "scr_data"
TABLE_PIX  = "pix_transacoes"
TABLES_IBGE = ["taxa_escolarizacao", "censo_demografico", "populacao_por_idade"]

PASTA = Path("/content/dados")
PASTA.mkdir(parents=True, exist_ok=True)

#Rodar com o localtunnel
URL_API_LOCAL = "https://safer-survivor-kerry-doctrine.trycloudflare.com"
HEADERS_BCB = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://olinda.bcb.gov.br/",
    "bypass-tunnel-reminder": "true"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS_BCB)

MAX_GET_RETRIES = 5
# [REMOVIDO] Constantes da leitura por .zip/.csv: CHUNK_SIZE, USECOLS_SCR, ano_anterior, URLS_SCR e SCR_LIMITE_DADOS
# (a leitura do SCR agora usa RENAME_SCR e URL_SCR_MES, definidos mais abaixo)

# [REMOVIDO] Colunas do CSV antigo (a API usa o RENAME_SCR)
# USECOLS_SCR = ["data_base", "uf", "porte", "cliente", "modalidade",
#     "carteira_inadimplencia", "carteira_vencida", "carteira_ativa", "vencido_15_a_60_dias", "vencido_61_a_90_dias", "vencido_acima_de_90_dias"]

# # [REMOVIDO] Anos e links dos .zip do BCB (a API usa o URL_SCR_MES)
# ano_anterior = datetime.now().year - 1
# URLS_SCR = [f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip" for ano in range(2021, ano_anterior+1)]

# [REMOVIDO] Limite de linhas do CSV antigo
# SCR_LIMITE_DADOS = 100000
# ano_anterior = datetime.now().year - 1
# URLS_SCR = [f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip" for ano in range(2021, ano_anterior+1)]

####ADICIONADO LUAN - Configurações da API do SCR por sub-região####

# [NOVO] Endereço base da API: tudo que vem antes do nome do endpoint
URL_BASE_SCR = "https://olinda.bcb.gov.br/olinda/servico/scr_sub_regiao/versao/v1/odata"

# [NOVO] Nome do endpoint (o recurso que vamos consultar)
ENDPOINT_SCR = "scr_sub_regiao"

# [NOVO] Tradutor: nome do campo na API -> nome da coluna que o notebook já usa
RENAME_SCR = {
    "DATA_BASE": "data_base",
    "CLIENTE": "cliente",
    "ESTADO": "uf",
    "SUB_REGIAO": "sub_regiao",                       # sub-região mantida conforme a task manda
    "MODALIDADE": "modalidade",
    "RISCO": "classe",                                # ATENÇÃO: substitui o antigo "porte"
    "CARTEIRA": "carteira_ativa",
    "VENCIDO_ACIMA_DE_15_DIAS": "carteira_vencida",   # ATENÇÃO: confirmar o significado
}

# [NOVO] Lista de campos que serão pedidos à API (parâmetro $select), vinda das chaves acima
CAMPOS_API_SCR = ",".join(RENAME_SCR.keys())

# [NOVO] Molde da URL de um mês. O <ANO_MES> é trocado depois por 202401, 202402...
URL_SCR_MES = (f"{URL_BASE_SCR}/{ENDPOINT_SCR}(DataBase=@DataBase)"
               f"?@DataBase=<ANO_MES>&$format=json&$select={CAMPOS_API_SCR}")

# [NOVO] Quantos meses para trás comparar no crescimento da carteira vencida (agingDivida)
# 12 = mesmo mês do ano anterior (tira a sazonalidade); 1 = mês anterior
PERIODOS_CRESCIMENTO = 12

# [REMOVIDO] NIVEIS_SALARIO (mapa de salário/porte para classe A a E): a API traz RISCO (AA-C, D-H, N/D) e não existe mais a coluna "porte"
# NIVEIS_SALARIO = {
#     # Salário Pessoa Física (PF)
#     "Acima de 20 salários mínimos": "A",
#     "Mais de 10 a 20 salários mínimos": "A",
#     "Mais de 5 a 10 salários mínimos": "B",
#     "Mais de 3 a 5 salários mínimos": "C",
#     "Mais de 2 a 3 salários mínimos": "D",
#     "Mais de 1 a 2 salários mínimos": "D",
#     "Até 1 salário mínimo": "E",
#     "Sem rendimento": "E",
#     "a": "A",

#     # Salário Pessoa Jurídica (PJ)
#     "Grande": "A",
#     "Médio": "B",
#     "Pequeno": "C",
#     "Micro": "D",
# }

PIX_LIMITE_DADOS = 35000
# [ALTERADO] Adicionado "&$orderby=AnoMes": sem isso a API devolvia as linhas em ordem qualquer e o $top cortava a maior parte do mês pedido
URL_PIX_API = f"https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?@DataBase='<ANO_MES>'&$top={PIX_LIMITE_DADOS}&$orderby=AnoMes&$format=json&$select=AnoMes,Estado_Ibge,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ"
LINKS_IBGE = {
    "taxa_escolarizacao": {
      "url": "https://apisidra.ibge.gov.br/values/t/7138/n3/all/v/10276/p/last%203/c2/6794/c58/100052,108866/d/v10276%201?formato=json",
      "use_cols": ["V", "D1N", "D3N"]
    },
    "censo_demografico": {
        "url": "https://apisidra.ibge.gov.br/values/t/4709/n1/all/n2/all/n3/all/v/all/p/all/d/v10605%202/l/p,v,t?formato=json",
        "use_cols": ["V" ,"D1N", "D2N", "D3N"]
    },
    "populacao_por_idade": {
        "url": "https://apisidra.ibge.gov.br/values/t/9514/n3/all/v/allxp/p/all/c2/6794/c287/6653,49108,49109,60040,60041,93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93095,93096,93097/c286/113635?formato=json",
        "use_cols": ["V", "D1N", "D3N", "D5N"]
    }
}

map_idade_media = {
    "0 a 4 anos": 2.0,
    "5 a 9 anos": 7.0,
    "10 a 14 anos": 12.0,
    "15 a 19 anos": 17.0,
    "20 a 24 anos": 22.0,
    "25 a 29 anos": 27.0,
    "30 a 34 anos": 32.0,
    "35 a 39 anos": 37.0,
    "40 a 44 anos": 42.0,
    "45 a 49 anos": 47.0,
    "50 a 54 anos": 52.0,
    "60 a 64 anos": 62.0,
    "65 a 69 anos": 67.0,
    "70 a 74 anos": 72.0,
    "80 a 84 anos": 82.0,
    "85 a 89 anos": 87.0,
    "90 a 94 anos": 92.0,
    "95 a 99 anos": 97.0,
    "100 anos ou mais": 100.0
}

---
#**Funções de Leitura**
---

In [ ]:
import time

def baixar_com_retry(url, SESSION, TIMEOUT):  #Leitura geral de arquivos
  primeria_tentativa = True
  headers_req = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
      "Accept": "application/json, text/plain, */*",
      "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
      "Referer": "https://olinda.bcb.gov.br/",
  }
  for i in range(1, MAX_GET_RETRIES + 1): #Faz get com limite de tentativas
    try:
      resposta = SESSION.get(url, headers=headers_req, timeout=TIMEOUT) #Tenta em um tempo curto
      resposta.raise_for_status()
      resposta.encoding = "utf-8-sig"
      return resposta
    except Exception as e:
      if i < MAX_GET_RETRIES:
        time.sleep(0.5 * i)
      else:
        print(f"Todas as {MAX_GET_RETRIES} tentativas falharam: {e}")
        return None


In [ ]:
def ler_json(url, SESSION, timeout = 5, chave = None, funcao_tratamento = None, filtro_colunas = None):
  resposta = baixar_com_retry(url, SESSION, timeout)
  if resposta is None:
    return

  json = resposta.json()

  if(chave and chave in json):
    df = pd.DataFrame(json[chave])
  else:
    df = pd.DataFrame(json)

  if filtro_colunas:
    df = df[filtro_colunas]
  if funcao_tratamento:
    return funcao_tratamento(df)

  return df

---
#**Função para enviar dados**
---

In [ ]:
def enviar(rota, dados):
    url = URL_API_LOCAL + rota
    for item in dados:
        try:
            resposta = SESSION.post(url, json=item, timeout=15)
            if resposta.status_code != 201:
                print(f"Erro {resposta.status_code} em {rota}: {resposta.text}")
                break
        except Exception as e:
            print(f"Erro: {e}")
            break

---
#**Leitura dados SCR**
---

In [ ]:
# [ALTERADO] Antes baixava um .zip e lia o CSV em pedaços. Agora faz uma chamada JSON por mês
def processar_url_scr(args):
    url, SESSION = args
    ano_mes = url.split("@DataBase=")[1][:6]     # [NOVO] só para mostrar em qual mês deu erro
    try:
        # [ALTERADO] usa o ler_json (que já usa o baixar_com_retry) no lugar do zipfile/read_csv
        return ler_json(url=url, SESSION=SESSION, timeout=30, chave="value")
    except Exception as e:
        print(f"SCR {ano_mes} falhou: {e}")      # [NOVO] um mês com problema não derruba os outros
        return None

def ler_scr(SESSION):
    # [ALTERADO] Antes: lista de anos (.zip). Agora: lista de meses (AAAAMM) desde 2021-01
    anos_meses = gerar_anos_meses_filtrado("2021-01-01")

    # [ALTERADO] Antes: URLs dos zips. Agora: a URL da API de cada mês
    urls = [URL_SCR_MES.replace("<ANO_MES>", ano_mes) for ano_mes in anos_meses]

    args = [(url, SESSION) for url in urls]       # [IGUAL]

    # [ALTERADO] max_workers=4: poucas chamadas ao mesmo tempo, para não sobrecarregar a API do BCB
    with ThreadPoolExecutor(max_workers=4) as executor:
        resultados = list(tqdm(executor.map(processar_url_scr, args), total=len(urls), desc="Baixando SCR..."))

    # [ALTERADO] Cada resultado agora é um DataFrame (ou None se o mês falhou), não uma lista deles
    dfs = [r for r in resultados if r is not None and not r.empty]
    if not dfs:                                   # [IGUAL]
        print("Nenhum dado do SCR encontrado.")
        return

    df = pd.concat(dfs, ignore_index=True)        # [IGUAL] junta todos os meses
    df = tratamento_scr_api(df)                   # [ALTERADO] trata uma vez só, depois de juntar
    print(f"Lido SCR: {df.shape[0]} linhas, {df.shape[1]} colunas")   # [IGUAL]
    salvar_parquet(df, TABLE_SCR)                 # [IGUAL] salva em scr_data.parquet

---
#**Leitura dados Pix**
---

In [ ]:
####ALTERADO LUAN - Agora aceita uma data de início (o Pix continua igual, pois o padrão é 2020-11-01)####

def gerar_anos_meses_filtrado(inicio="2020-11-01"):     # [ALTERADO] ganhou o parâmetro "inicio"
    data_atual = datetime.now()
    anos_meses = pd.date_range(start=inicio, end=data_atual, freq="MS")   # [ALTERADO] usa "inicio" no lugar da data fixa
    lista_completa = anos_meses[:-1].strftime("%Y%m").tolist()            # [IGUAL] tira o mês atual e formata como "202101"

    return lista_completa

In [ ]:
def processar_url_pix(args):
    url, SESSION = args
    # [NOVO] Mês que esta chamada pediu (vem da própria URL: @DataBase='202609')
    ano_mes = url.split("@DataBase='")[1][:6]

    # [ALTERADO] Antes: return ler_json(...) devolvia tudo que a API mandava (mês pedido + seguintes)
    df = ler_json(url=url, SESSION=SESSION, timeout=10, chave="value", funcao_tratamento=tratamento_pix)
    if df is None:                                     # [NOVO] segurança, igual ao resto do notebook
        return None

    # [NOVO] Fica só com o mês pedido, descartando os meses extras que a API manda junto
    return df[df["AnoMes"].astype(str) == ano_mes]
def ler_pix(SESSION):
    anos_meses = gerar_anos_meses_filtrado()

    URLS_PIX = [URL_PIX_API.replace("<ANO_MES>", ano_mes) for ano_mes in anos_meses]

    args = [(url, SESSION) for url in URLS_PIX]
    with ThreadPoolExecutor(max_workers=4) as executor:   # [ALTERADO] antes: ThreadPoolExecutor()
        resultado = list(tqdm(executor.map(processar_url_pix, args), total=len(URLS_PIX), desc="Baixando arquivos pix..."))

    if not any(r is not None and not r.empty for r in resultado):
        print("Nenhum dado do PIX encontrado.")
        return

    df = pd.concat([r for r in resultado if r is not None and not r.empty], ignore_index=True)
    print(f"Lido PIX: {df.shape[0]} linhas, {df.shape[1]} colunas")
    salvar_parquet(df, TABLE_PIX)

---
#**Leitura dados IBGE**
---

In [ ]:
def ler_urls_ibge(SESSION):
    funcoes_tratamento_ibge = [tratamento_taxa_escolarizacao, tratamento_censo_demografico, tratamento_populacao_por_idade]
    tratamentos = {k: v for k, v in zip(TABLES_IBGE, funcoes_tratamento_ibge)}

    for table_name, item in LINKS_IBGE.items():
        print(f"Baixando {table_name}...")
        df = ler_json(item["url"], SESSION=SESSION, timeout=10, filtro_colunas=item["use_cols"], funcao_tratamento=tratamentos[table_name])

        if df is not None and not df.empty:
            salvar_parquet(df, table_name)
        else:
            print(f"Nenhum dado encontrado para {table_name}.")

---
#**Tratamento**
---

In [ ]:
#Funções gerais de tratamento de dados

def limpeza_string(df, colunas=None):
    if colunas is None:
        colunas = df.select_dtypes(include="object").columns.tolist()
    else:
        colunas = [col for col in colunas if col in df.columns and df[col].dtype == "object"]

    if not colunas:
        return df

    for col in colunas:
        df[col] = df[col].str.strip()
    return df

def converter_numerico(df, *colunas):
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def limpar_nulos(df):
    numericas = df.select_dtypes(include="number").columns
    df[numericas] = df[numericas].fillna(df[numericas].median())

    categoricas = df.select_dtypes(include="object").columns
    df[categoricas] = df[categoricas].fillna("NAO_INFORMADO")
    return df

def formatar_data(df, colunas, formato="%Y-%m-%d"):
    if not colunas: return df
    if isinstance(colunas, str):
        colunas = [colunas]
    for col in colunas:
        df[col] = pd.to_datetime(df[col], format=formato).dt.strftime("%Y%m")
    return df

def formatar_numero(df, colunas):
    if isinstance(colunas, str):
        colunas = [colunas]
    for col in colunas:
        df[col] = pd.to_numeric(df[col].str.replace(",", "."), errors="coerce")
    return df

# [REMOVIDO] def converter_nivel_salario(df, coluna): ... (dependia do NIVEIS_SALARIO; não é mais chamada)
# def converter_nivel_salario(df, coluna):
#   df[coluna] = df[coluna].map(NIVEIS_SALARIO)
#   return df

def converter_uf(df, coluna):
  estados_brasil = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF", "Espírito Santo": "ES",
    "Goiás": "GO", "Maranhão": "MA", "Mato Grosso": "MT", "Mato Grosso do Sul": "MS",
    "Minas Gerais": "MG", "Pará": "PA", "Paraíba": "PB", "Paraná": "PR",
    "Pernambuco": "PE", "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN",
    "Rio Grande do Sul": "RS", "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC",
    "São Paulo": "SP", "Sergipe": "SE", "Tocantins": "TO",
    11: "RO", 12: "AC", 13: "AM", 14: "RR",
    15: "PA", 16: "AP", 17: "TO", 21: "MA",
    22: "PI", 23: "CE", 24: "RN", 25: "PB",
    26: "PE", 27: "AL", 28: "SE", 29: "BA",
    31: "MG", 32: "ES", 33: "RJ", 35: "SP",
    41: "PR", 42: "SC", 43: "RS", 50: "MS",
    51: "MT", 52: "GO", 53: "DF"
  }

  df = df.rename(columns = {
      coluna: "uf"
  })
  df["uf"] = df["uf"].map(estados_brasil)

  return df

def converter_uf_para_macro_regiao(df, coluna):
  mapeamento_regioes = {
      'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
      'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
      'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
      'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
      'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
      'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
  }

  df['regiao'] = df[coluna].map(mapeamento_regioes)

  return df

def converter_porcentagem(df, coluna):
  df[coluna] = pd.to_numeric(df[coluna], errors='coerce') / 100
  return df

In [ ]:
## [REMOVIDO] def tratamento_scr(df): ... return df   (era o tratamento do CSV, trocado pelo tratamento_scr_api)

####ADICIONADO LUAN - Tratamento dos dados vindos da API (diferente do que vinha do CSV)####

def tratamento_scr_api(df):
    # [NOVO] Troca os nomes da API pelos nomes do notebook (usa o RENAME_SCR das Configurações)
    # .copy() evita avisos do pandas ao alterar o DataFrame depois
    df = df.rename(columns=RENAME_SCR).copy()

    # [ALTERADO] Mesma limpeza de espaços de antes, com as colunas de texto da API
    df = limpeza_string(df, ["cliente", "uf", "sub_regiao", "classe", "modalidade"])

    # [NOVO] A API manda classe nula (None) em vários meses; vira "N/D" (igual ao que a API já usa)
    # Sem isso, o groupby do padronizar_df_scr descartaria essas linhas e a carteira somaria menos
    df["classe"] = df["classe"].fillna("N/D")

    # [ALTERADO] A API manda 202401 como número inteiro; aqui vira texto "202401"
    # Antes: formatar_data convertia "2024-01-01" em "202401" (daria erro com os dados da API)
    df["data_base"] = df["data_base"].astype(str)

    # [NOVO] "NI" = estado não identificado. Não existe no mapa de regiões e viraria regiao = NaN
    df = df[df["uf"] != "NI"].copy()

    # [IGUAL] Cria a coluna "regiao" (Norte, Sul...) a partir da uf
    df = converter_uf_para_macro_regiao(df, "uf")

    # [ALTERADO] Antes: formatar_numero (usava .str.replace, que só funciona com texto)
    # Agora os valores já chegam como float, então só garantimos o tipo numérico
    df = converter_numerico(df, "carteira_ativa", "carteira_vencida")

    # [IGUAL] Valor ausente vira R$ 0,00
    df[["carteira_ativa", "carteira_vencida"]] = df[["carteira_ativa", "carteira_vencida"]].fillna(0.0)

    # [REMOVIDO] limpar_nulos, converter_uf e converter_nivel_salario:
    #   a API já traz a sigla da UF e não existe mais a coluna "porte"
    return df

In [ ]:
def tratamento_pix(df):    # [ALTERADO] estava comentada com "#"; o ler_pix precisa dela ativa
  df = limpar_nulos(df)    # [IGUAL] daqui para baixo nada muda
  df = limpeza_string(df, ["AnoMes"])
  df = converter_numerico(df, "VL_PagadorPF", "QT_PagadorPF", "VL_PagadorPJ", "QT_PagadorPJ", "QT_PES_PagadorPF", "QT_PES_PagadorPJ")
  df = converter_uf(df, "Estado_Ibge")
  df = converter_uf_para_macro_regiao(df, "uf")
  return df

In [ ]:
def tratamento_taxa_escolarizacao(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["D1N", "D3N"])
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = converter_porcentagem(df, "V")
  df = df.rename(columns={"D3N": "ano", "V": "valor"})
  return df

def tratamento_censo_demografico(df):
  df = limpar_nulos(df)
  df["V"] = df["V"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
  df = converter_numerico(df, "V")
  df = limpeza_string(df, ["D1N", "D3N"])
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = df.rename(columns={"D3N": "ano", "V": "valor"})
  return df

def tratamento_populacao_por_idade(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["D1N", "D3N", "D5N"])
  df = converter_numerico(df, "V")
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = df.rename(columns={"D3N": "ano", "V": "valor", "D5N": "idade"})
  df["idade"] = df["idade"].map(map_idade_media)
  return df

---
#**Perpetuação**
---

In [ ]:
def salvar_parquet(df, nome):
  caminho = PASTA / f"{nome}.parquet"
  df.to_parquet(caminho, index=False)
  print(f" --Salvo em: {caminho}")

# **Refatoração**

In [ ]:
def validar_consistencia_saldos(df):
    """Função para checar se a soma dos saldos por modalidade é igual à carteira ativa total."""
    # 1. Filtra registros de carteira ativa
    df_ativa = df[df["metrica"] == "carteira_ativa"]

    # 2. Agrupa por modalidade e calcula a soma total
    soma_modalidades = (
        df_ativa.groupby(["ano_mes", "uf", "modalidade"])["valor"]
        .sum()
        .groupby(["ano_mes", "uf"])
        .sum()
        .reset_index(name="soma_modalidades")
    )

    # 3. Obtém o total da carteira ativa informado para o período/UF
    total_carteira = (
        df_ativa.groupby(["ano_mes", "uf"])["valor"]
        .sum()
        .reset_index(name="total_carteira_ativa")
    )

    # 4. Cruza e valida a diferença
    checagem = pd.merge(soma_modalidades, total_carteira, on=["ano_mes", "uf"])
    checagem["diferenca"] = (
        checagem["soma_modalidades"] - checagem["total_carteira_ativa"]
    ).round(2)

    inconsistencias = checagem[checagem["diferenca"] != 0]

    if not inconsistencias.empty:
        print(
            f"[VALIDAÇÃO] Encontradas {len(inconsistencias)} inconsistências entre modalidades e carteira ativa!"
        )
    else:
        print(
            "[VALIDAÇÃO] Saldos entre modalidades e carteira ativa estão 100% consistentes!"
        )

def padronizar_df_scr(df_scr):
    df_scr = df_scr.rename(columns={"data_base": "ano_mes"})                      # [IGUAL]

    df_scr["tipo"] = df_scr["cliente"].map({"PF": "pf"}).fillna("pj")             # [IGUAL]

    df_scr = df_scr.melt(
        id_vars=["ano_mes", "regiao", "uf", "sub_regiao", "tipo", "classe", "modalidade"],   # [ALTERADO] + "sub_regiao"
        value_vars=["carteira_vencida", "carteira_ativa"],                        # [IGUAL]
        var_name="metrica",
        value_name="valor",
    )
    df_scr["origem"] = "scr"                                                      # [IGUAL]

     # [REMOVIDO] validar_consistencia_saldos(df_scr): comparava a soma do dado com ela mesma e rodava por chunk (mensagem repetida)

    for col in ["tipo", "classe", "origem", "metrica", "uf", "sub_regiao", "modalidade"]:   # [ALTERADO] + "sub_regiao"
        if col in df_scr.columns:
            df_scr[col] = df_scr[col].astype("category")

    df_scr = (
        df_scr.groupby(
            [
                "ano_mes",
                "regiao",
                "uf",
                "sub_regiao",                                                     # [NOVO]
                "tipo",
                "classe",
                "modalidade",
                "metrica",
                "origem",
            ],
            observed=True,
        )["valor"]
        .sum()
        .reset_index()
    )

    return df_scr[[
        "ano_mes",
        "regiao",
        "uf",
        "sub_regiao",                                                             # [NOVO]
        "tipo",
        "classe",
        "metrica",
        "origem",
        "valor",
    ]]

In [ ]:
def validar_scr_contra_origem(caminho_origem="/content/dados/scr_data.parquet",
                              caminho_final="/content/dados/scr_pix.parquet"):   # [NOVO]
    origem = pd.read_parquet(caminho_origem)
    origem["data_base"] = origem["data_base"].astype(str)
    esperado = origem.groupby("data_base")[["carteira_ativa", "carteira_vencida"]].sum()

    final = pd.read_parquet(caminho_final)
    final = final[final["origem"] == "scr"]
    obtido = final.groupby(["ano_mes", "metrica"], observed=True)["valor"].sum().unstack()

    dif = (esperado - obtido[["carteira_ativa", "carteira_vencida"]]).abs()
    erros = dif[(dif > 1).any(axis=1)]

    if erros.empty:
        print(f"[VALIDAÇÃO] SCR: soma da origem = soma do scr_pix em {len(esperado)} meses.")
    else:
        print(f"[VALIDAÇÃO] SCR: divergência em {len(erros)} meses (diferença em R$):")
        print(erros)

In [ ]:
# [REMOVIDO] def padronizar_df_pix(df_pix): ... (cópia antiga da seção Refatoração)
# Ela gerava as colunas extras "modalidade" e "categoria_credito", mas era sobrescrita
# pela versão da seção Scoring, que é a que de fato roda. Fica só a versão da seção Scoring.
# import numpy as np
# import pandas as pd


# def padronizar_df_pix(df_pix):
#     df_pix = df_pix.rename(columns={"AnoMes": "ano_mes"})

#     df_pix = df_pix.melt(
#         id_vars=["ano_mes", "regiao", "uf"],
#         value_vars=[
#             "VL_PagadorPF",
#             "QT_PagadorPF",
#             "VL_PagadorPJ",
#             "QT_PagadorPJ",
#             "QT_PES_PagadorPF",
#             "QT_PES_PagadorPJ",
#         ],
#         var_name="metrica_raw",
#         value_name="valor",
#     )

#     df_pix["tipo"] = (
#         df_pix["metrica_raw"].str.endswith("PF").map({True: "pf", False: "pj"})
#     )

#     df_pix["metrica"] = np.select(
#         [
#             df_pix["metrica_raw"].str.contains("VL"),
#             df_pix["metrica_raw"].str.contains("QT_PES"),
#             df_pix["metrica_raw"].str.contains("QT"),
#         ],
#         ["vl_pagador", "qt_pes_pagador", "qt_pagador"],
#         default="outros",
#     )

#     df_pix["origem"] = "pix"

#     df_pix = (
#         df_pix.groupby(
#             ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
#             observed=True,
#         )["valor"]
#         .sum()
#         .reset_index()
#     )

#     # Compatibilidade de colunas com o schema do SCR
#     df_pix["classe"] = pd.NA
#     df_pix["modalidade"] = "pix"
#     df_pix["categoria_credito"] = "nao_aplica"

#     for col in [
#         "tipo",
#         "origem",
#         "metrica",
#         "uf",
#         "regiao",
#         "classe",
#         "modalidade",
#         "categoria_credito",
#     ]:
#         if col in df_pix.columns:
#             df_pix[col] = df_pix[col].astype("category")

#     return df_pix[[
#         "ano_mes",
#         "regiao",
#         "uf",
#         "tipo",
#         "classe",
#         "modalidade",
#         "categoria_credito",
#         "metrica",
#         "origem",
#         "valor",
#     ]]

---
#**Pré Processamento**
---

In [ ]:
def processar():
    print(f"--- Iniciando Extração de Dados ---")

    SESSION = requests.Session()
    SESSION.headers.update(HEADERS_BCB)

    try:
        ler_scr(SESSION)
    except Exception as e:
        print(f"Erro ao ler SCR: {e}")

    try:
        ler_pix(SESSION)
    except Exception as e:
        print(f"Erro ao ler Pix: {e}")

    try:
        ler_urls_ibge(SESSION)
    except Exception as e:
        print(f"Erro ao ler IBGE: {e}")

---
#**Scoring**
---

In [ ]:
def extract_data(path:str):
    df = pd.read_parquet(path)
    return df

Leitura Parquets Pix e Scr
---

In [ ]:
# [REMOVIDO] def padronizar_df_scr(df_scr): ... (cópia antiga da seção Scoring)
# Fica só a versão da seção Refatoração, já ajustada para as colunas da API

In [ ]:
def padronizar_df_pix(df_pix):
    df_pix = df_pix.rename(columns={"AnoMes": "ano_mes"})

    df_pix = df_pix.melt(
        id_vars=["ano_mes", "regiao", "uf"],
        value_vars=["VL_PagadorPF", "QT_PagadorPF", "VL_PagadorPJ", "QT_PagadorPJ", "QT_PES_PagadorPF", "QT_PES_PagadorPJ"],
        var_name="metrica_raw",
        value_name="valor"
    )

    df_pix["tipo"] = df_pix["metrica_raw"].str.endswith("PF").map({True: "pf", False: "pj"})

    df_pix["metrica"] = np.select(
        [df_pix["metrica_raw"].str.contains("VL"),
         df_pix["metrica_raw"].str.contains("QT_PES"),
         df_pix["metrica_raw"].str.contains("QT")],
        ["vl_pagador", "qt_pes_pagador", "qt_pagador"],
        default="outros"
    )

    df_pix["origem"] = "pix"

    df_pix = df_pix.groupby(
        ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
        observed=True
    )["valor"].sum().reset_index()

    df_pix["classe"] = pd.NA  # adiciona depois do groupby

    for col in ["tipo", "origem", "metrica", "uf", "regiao"]:
        df_pix[col] = df_pix[col].astype("category")

    return df_pix[["ano_mes", "regiao", "uf", "tipo", "classe", "metrica", "origem", "valor"]]

In [ ]:
def construir_df_scr_pix():
  if not Path('/content/dados/pix_transacoes.parquet').exists() or not Path('/content/dados/scr_data.parquet').exists():
      raise FileNotFoundError("Arquivos pix_transacoes.parquet ou scr_data.parquet não foram gerados pela API do BCB. Verifique a conexão com a API do Banco Central na etapa anterior.")

  chunks_pix = []
  chunks_scr = []

  #Padroniza o parquet pix por chunk
  tabela_pix = pq.ParquetFile('/content/dados/pix_transacoes.parquet')

  for batch in tabela_pix.iter_batches(batch_size=100000):
    chunks_pix.append(padronizar_df_pix(batch.to_pandas()))
  df_pix = pd.concat(chunks_pix, ignore_index=True)

  df_pix = df_pix.groupby(                                                   # [IGUAL] groupby do Pix (sem sub_regiao)
      ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
      observed=True
  )["valor"].sum().reset_index()
  df_pix["classe"] = pd.NA                                                   # [IGUAL]
  df_pix["sub_regiao"] = pd.NA                                               # [NOVO]

  #Padroniza o parquet scr por chunk
  tabela_scr = pq.ParquetFile('/content/dados/scr_data.parquet')

  for batch in tabela_scr.iter_batches(batch_size=100000):
    chunks_scr.append(padronizar_df_scr(batch.to_pandas()))
  df_scr = pd.concat(chunks_scr, ignore_index=True)

  df_scr = df_scr.groupby(
      ["ano_mes", "regiao", "uf", "sub_regiao", "tipo", "classe", "metrica", "origem"],   # [ALTERADO] + "sub_regiao"
      observed=True
  )["valor"].sum().reset_index()

  df_pix["ano_mes"] = df_pix["ano_mes"].astype(str)                          # [IGUAL]
  df_scr["ano_mes"] = df_scr["ano_mes"].astype(str)                          # [IGUAL]

  #une os dois parquets em um individual
  df_scr_pix = pd.concat([df_scr, df_pix], ignore_index=True)                # [IGUAL]

  salvar_parquet(df_scr_pix, "scr_pix")                                      # [IGUAL]

Leitura Parquets IBGE
---

In [ ]:
def calculo_idade():
  #---populacao por idade
  df_idade = pd.read_parquet('/content/dados/populacao_por_idade.parquet')

  df_idade["idade_ponderada"] = df_idade["idade"] * df_idade["valor"]

  df_idade = (
      df_idade.groupby(["ano", "regiao", "uf"])
      .agg(
          soma_idade=("idade_ponderada", "sum"),
          soma_pop=("valor", "sum")
      )
      .reset_index()
  )

  df_idade["idade_media"] = df_idade["soma_idade"] / df_idade["soma_pop"]

  df_idade["idade_max"] = df_idade.groupby("ano")["idade_media"].transform("max")

  df_idade["bonusDemografico"] = df_idade["idade_max"] - df_idade["idade_media"]

  return df_idade[["ano", "regiao", "uf", "bonusDemografico"]]

In [ ]:
def construir_df_ibge():
  #---escolarizacao
  df_escolarizacao = extract_data('/content/dados/taxa_escolarizacao.parquet')

  df_escolarizacao = df_escolarizacao.groupby(  #realiza a média de valores com mesmo ano, regiao e uf
      ["ano", "regiao", "uf"],
      observed=True)["valor"].mean().reset_index()
  df_escolarizacao = df_escolarizacao.rename(columns={"valor": "taxa_escolarizacao"})

  #---censo demografico
  df_censo_demografico = extract_data('/content/dados/censo_demografico.parquet')

  #separa colunas por metrica
  df_censo_demografico = df_censo_demografico.pivot_table(
      index=["ano", "regiao", "uf"],
      columns="D2N",
      values="valor").reset_index()

  df_censo_demografico = df_censo_demografico.rename(columns={
      "População residente": "populacao_residente",
      "Taxa de crescimento geométrico": "taxa_crescimento",
      "Variação absoluta da população residente 2010 compatibilizada": "variacao_populacao"
  })

  df_idade = calculo_idade()

  #arruma porcentagem
  df_censo_demografico["taxa_crescimento"] = df_censo_demografico["taxa_crescimento"]/100

  df_censo_demografico.columns.name = None

  # [NOVO] Censo e idade só existem em anos de censo (2010, 2022), então não casam com os anos da escolarização.
  # Fica com o dado mais recente de cada UF (.last()) e tira a coluna "ano", para valer para todos os anos
  df_censo_demografico = (df_censo_demografico.sort_values("ano")
      .groupby(["regiao", "uf"]).last().reset_index().drop(columns=["ano"]))
  df_idade = (df_idade.sort_values("ano")
      .groupby(["regiao", "uf"]).last().reset_index().drop(columns=["ano"]))

  # [ALTERADO] Antes cruzava por ["ano", "regiao", "uf"]. Agora cruza só por ["regiao", "uf"]
  df_ibge = df_escolarizacao.merge(
      df_censo_demografico,
      on=["regiao", "uf"],
      how="left").merge(
      df_idade,
      on=["regiao", "uf"],
      how="left")

  df_ibge = (
      df_ibge.sort_values(["uf", "regiao", "ano"])
      .ffill()
      .reset_index(drop=True)   # [ALTERADO] antes: .reset_index()
  )
  salvar_parquet(df_ibge, "ibge")

In [ ]:
def normalizacao(df, coluna):
  df[coluna + "_max"] = df.groupby("ano_mes")[coluna].transform("max")
  df[coluna + "_min"] = df.groupby("ano_mes")[coluna].transform("min")
  df[coluna] = (1 + (df[coluna] - df[coluna + "_min"]) / (df[coluna + "_max"] - df[coluna + "_min"]) * 4).round(2)
  df = df.drop(columns=[coluna + "_max", coluna + "_min"])

  return df

---
#**Eixo I – Risco de Crédito**
---

In [ ]:
filtro = ["ano_mes", "regiao", "uf"]


def agrupar_metrica(df, metrica, filtro_extra=None):
    mask = df["metrica"] == metrica
    if filtro_extra is not None:
        mask &= filtro_extra
    return df[mask].groupby(filtro)["valor"].sum()

In [ ]:
def calcular_crescimento_vencida(df_scr_pix_base):
    base = df_scr_pix_base[df_scr_pix_base["metrica"] == "carteira_vencida"]
    atual = (
        base.groupby(["ano_mes", "regiao", "uf"], observed=True)["valor"]
        .sum()
        .reset_index()
    )
    atual = atual.rename(columns={"valor": "vencida"})

    atual["ano_mes"] = atual["ano_mes"].astype(str)
    atual["uf"] = atual["uf"].astype(str)
    atual["regiao"] = atual["regiao"].astype(str)
    atual["data"] = pd.to_datetime(atual["ano_mes"], format="%Y%m")

    passado = atual[["uf", "data", "vencida"]].rename(
        columns={"vencida": "vencida_anterior"}
    )
    passado["data"] = passado["data"] + pd.DateOffset(
        months=PERIODOS_CRESCIMENTO
    )

    atual = atual.merge(passado, on=["uf", "data"], how="left")

    atual["agingDivida"] = np.where(
        atual["vencida_anterior"] > 0,
        (atual["vencida"] / atual["vencida_anterior"]) - 1,
        0.0,
    )
    return atual[["ano_mes", "regiao", "uf", "agingDivida"]]


def calcular_eixo_i(df_scr_pix_base, df_ibge_base):
    df_scr_pix = df_scr_pix_base.copy()
    df_ibge = df_ibge_base.copy()

    df_crescimento = calcular_crescimento_vencida(df_scr_pix_base)

    df_scr_pix["ano"] = df_scr_pix["ano_mes"].astype(str).str[:4]

    df_calculos = df_scr_pix.merge(
        df_ibge[["ano", "regiao", "uf", "taxa_escolarizacao"]],
        on=["ano", "regiao", "uf"],
        how="left",
    ).drop(columns=["ano"])
        # [NOVO] Ordena por UF e mês: o ffill/bfill abaixo preenche conforme a ordem das linhas.
    # Sem isso, 2021 e 2022 receberiam a escolarização de 2025 em vez da de 2023
    df_calculos = df_calculos.sort_values(["uf", "ano_mes"])

    df_calculos["taxa_escolarizacao"] = df_calculos.groupby("uf")[
        "taxa_escolarizacao"
    ].transform(lambda x: x.ffill().bfill())

    # --- CORREÇÃO 1: Removida a duplicação do agrupar_metrica da carteira_ativa_classes_de ---   # [IGUAL]
    df_metricas = pd.concat(
        [
            agrupar_metrica(df_calculos, "carteira_vencida").rename(
                "carteira_vencida"
            ),                                                                                    # [IGUAL]
            agrupar_metrica(df_calculos, "carteira_ativa").rename(
                "carteira_ativa"
            ),                                                                                    # [IGUAL]

            # [NOVO] DECISÃO (21/10/2026): a API do SCR traz RISCO de crédito (AA-C, D-H, N/D),
            # e não mais a classe de renda (A a E). Por isso "D-H" (risco alto) passa a valer
            # no lugar de "D" e "E". O nome carteira_ativa_classes_de e o indicador
            # fragilidadeRenda foram mantidos para não quebrar o resto do código.
            agrupar_metrica(
                df_calculos, "carteira_ativa", df_calculos["classe"].isin(["D-H"])   # [ALTERADO] antes: .isin(["D", "E"])
            ).rename("carteira_ativa_classes_de"),
        ],
        axis=1,
    ).reset_index()                                                                               # [IGUAL]
    df_calculos = (
        df_calculos[["ano_mes", "regiao", "uf", "taxa_escolarizacao"]]
        .drop_duplicates()
        .merge(df_metricas, on=filtro, how="left")
    )
      # [NOVO] Remove meses que só existem no Pix (202011, 202012, 202609): sem SCR não há carteira
    # e esses meses apareceriam com zeros falsos
    df_calculos = df_calculos.dropna(subset=["carteira_ativa"])

    df_calculos = df_calculos.merge(df_crescimento, on=filtro, how="left")
    df_calculos["agingDivida"] = df_calculos["agingDivida"].fillna(0.0)

    # --- Divisões com tratamento de divisão por zero ---
    df_calculos["inadimplenciaReal"] = np.where(
        df_calculos["carteira_ativa"] > 0,
        df_calculos["carteira_vencida"] / df_calculos["carteira_ativa"],
        0.0,
    )

    # [NOVO] DECISÃO (21/10/2026): a partir de 202501 a API só envia o risco "N/D" (não há "D-H").
    # Nesses meses todas as UFs ficam com fração 0, a normalizacao gera NaN (0/0) e o fillna(0)
    # final transforma em 0. A escala válida é de 1 a 5, então 0 significa "SEM DADO", e não "risco nulo".
    df_calculos["fragilidadeRenda"] = np.where(                                  # [IGUAL]
        df_calculos["carteira_ativa"] > 0,
        df_calculos["carteira_ativa_classes_de"] / df_calculos["carteira_ativa"],
        0.0,
    )

    # --- CORREÇÃO 2: Sintaxe de vulnerabilidadeSocial corrigida ---
    df_calculos["vulnerabilidadeSocial"] = 1 - df_calculos["taxa_escolarizacao"]

    # --- Normalização ---
    df_calculos = normalizacao(df_calculos, "inadimplenciaReal")
    df_calculos = normalizacao(df_calculos, "fragilidadeRenda")
    df_calculos = normalizacao(df_calculos, "agingDivida")
    df_calculos = normalizacao(df_calculos, "vulnerabilidadeSocial")

    # --- Limpeza final ---
    df_calculos = df_calculos.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_calculos[[
        "ano_mes",
        "regiao",
        "uf",
        "inadimplenciaReal",
        "fragilidadeRenda",
        "agingDivida",
        "vulnerabilidadeSocial",
    ]]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configurações ---
PERIODO = None

def obter_mapeamento_subregioes():
    # Mapeamento oficial de códigos da API do BCB para nomes
    return {
        "70": "Distrito Federal e Entorno",
        "71": "Goiás Central / Goiânia",
        "72": "Sul / Sudoeste Goiano",
        "73": "Norte / Nordeste Goiano",
        "74": "Cuiabá e Baixada Cuiabana",
        "75": "Norte / Sudoeste Mato-Grossense",
        "76": "Sul / Sudeste Mato-Grossense",
        "78": "Campo Grande e Centro-Sul MS",
        "79": "Leste / Pantanal MS",
        "20": "Metropolitana de Salvador",
        "21": "Recôncavo / Sul Baiano",
        "22": "Oeste / São Francisco Baiano",
        "23": "Centro-Norte Baiano",
        "24": "Grande Fortaleza",
        "25": "Sertão / Cariri Cearense",
        "26": "Região de Sobral / Norte Cearense",
        "27": "Metropolitana de Recife",
        "28": "Agreste / Sertão Pernambucano",
        "29": "Metropolitana de São Luís",
        "30": "Sul / Oeste Maranhense",
        "31": "Grande Teresina",
        "32": "Sul / Sudeste Piauiense",
        "33": "Metropolitana de Natal",
        "34": "Oeste / Central Potiguar",
        "35": "Metropolitana de João Pessoa",
        "36": "Campina Grande e Sertão Paraibano",
        "37": "Metropolitana de Maceió",
        "38": "Sertão / Agreste Alagoano",
        "39": "Metropolitana de Aracaju e Interior SE",
        "50": "Grande São Paulo",
        "51": "Campinas / Sorocaba / Piracicaba",
        "52": "Ribeirão Preto / Franca / São Carlos",
        "53": "São José do Rio Preto / Araçatuba",
        "54": "Bauru / Marília / Presidente Prudente",
        "55": "Vale do Paraíba / Litoral Norte",
        "56": "Santos / Litoral Paulista",
        "57": "Metropolitana de Belo Horizonte",
        "58": "Triângulo Mineiro / Alto Paranaíba",
        "59": "Sul / Sudoeste de Minas",
        "60": "Zona da Mata / Campo das Vertentes",
        "61": "Norte de Minas / Vale do Jequitinhonha",
        "62": "Vale do Rio Doces / Mucuri",
        "63": "Grande Vitória / Litoral Capixaba",
        "64": "Sul / Serrana Capixaba",
        "65": "Metropolitana do Rio de Janeiro",
        "66": "Sul / Serrana Fluminense",
        "67": "Norte / Noroeste Fluminense",
        "80": "Grande Curitiba",
        "81": "Londrina / Maringá / Norte Paranaense",
        "82": "Cascavel / Foz / Oeste Paranaense",
        "83": "Ponta Grossa / Campos Gerais",
        "84": "Metropolitana de Porto Alegre",
        "85": "Caxias do Sul / Serra Gaúcha",
        "86": "Passo Fundo / Noroeste Gaúcho",
        "87": "Pelotas / Santa Maria / Sul Gaúcho",
        "88": "Grande Florianópolis",
        "89": "Joinville / Planalto Norte Catarinense",
        "90": "Blumenau / Vale do Itajaí",
        "91": "Chapecó / Oeste Catarinense",
        "92": "Criciúma / Sul Catarinense",
        "10": "Metropolitana de Belém / Nordeste Paraense",
        "11": "Marabá / Santarém / Sul-Oeste Paraense",
        "12": "Metropolitana de Manaus / Interior AM",
        "13": "Porto Velho e Rondônia",
        "14": "Macapá e Amapá",
        "15": "Boa Vista e Roraima",
        "16": "Rio Branco e Acre",
        "17": "Palmas e Tocantins",
    }

def plotar_eixo_i_regioes_subregioes(df_scr_pix=None, df_ibge=None, periodo_ref=None,
                                     subregiao_por_uf=False, usar_nomes_subregiao=True,
                                     mostrar_graficos=True):
    indicadores = [
        ("inadimplenciaReal", "1. Inadimplência Real", "#2b90c8"),
        ("fragilidadeRenda", "2. Fragilidade de Renda", "#3ca86e"),
        ("agingDivida", "3. Aging da Dívida", "#9467eb"),
        ("vulnerabilidadeSocial", "4. Vulnerabilidade Social", "#f09f2d"),
    ]
    cols_ind = [c for c, _, _ in indicadores]
    sub_invalidas = {"nan", "None", "", "0"}

    def escolarizacao_uf_mes(df_meses_uf):
        base = df_meses_uf[["ano_mes", "uf"]].drop_duplicates().copy()
        base["ano"] = base["ano_mes"].str[:4]
        ibge = df_ibge[["ano", "uf", "taxa_escolarizacao"]].copy()
        ibge["ano"] = ibge["ano"].astype(str)
        ibge["uf"] = ibge["uf"].astype(str)
        ibge["taxa_escolarizacao"] = ibge["taxa_escolarizacao"].where(ibge["taxa_escolarizacao"] > 0)
        ibge = ibge.groupby(["ano", "uf"], as_index=False)["taxa_escolarizacao"].mean()
        out = base.merge(ibge, on=["ano", "uf"], how="left").sort_values(["uf", "ano_mes"])
        out["taxa_escolarizacao"] = out.groupby("uf")["taxa_escolarizacao"].transform(lambda x: x.ffill().bfill())
        return out[["ano_mes", "uf", "taxa_escolarizacao"]]

    def calcular_nivel(nivel):
        chave = ["ano_mes"] + nivel
        scr = df_scr_pix[
            (df_scr_pix["origem"].astype(str) == "scr")
            & (df_scr_pix["metrica"].astype(str).isin(["carteira_ativa", "carteira_vencida"]))
        ].copy()
        for c in ["ano_mes", "regiao", "uf", "sub_regiao", "classe", "metrica"]:
            scr[c] = scr[c].astype(str)
        scr["valor" ] = pd.to_numeric(scr["valor"], errors="coerce").fillna(0.0)

        if "sub_regiao" in nivel:
            scr = scr[~scr["sub_regiao"].isin(sub_invalidas)]

        ativa = scr[scr["metrica"] == "carteira_ativa"]
        vencida = scr[scr["metrica"] == "carteira_vencida"]
        ativa_dh = ativa[ativa["classe"] == "D-H"]

        df = pd.concat([
            ativa.groupby(chave)["valor"].sum().rename("carteira_ativa"),
            vencida.groupby(chave)["valor"].sum().rename("carteira_vencida"),
            ativa_dh.groupby(chave)["valor"].sum().rename("carteira_ativa_dh"),
        ], axis=1).fillna(0.0).reset_index()

        df["data"] = pd.to_datetime(df["ano_mes"], format="%Y%m")
        passado = df[nivel + ["data", "carteira_vencida"]].rename(columns={"carteira_vencida": "vencida_anterior"})
        passado["data"] = passado["data"] + pd.DateOffset(months=PERIODOS_CRESCIMENTO)
        df = df.merge(passado, on=nivel + ["data"], how="left")
        df["agingDivida"] = np.where(df["vencida_anterior"] > 0, df["carteira_vencida"] / df["vencida_anterior"] - 1, 0.0)

        df["inadimplenciaReal"] = np.where(df["carteira_ativa"] > 0, df["carteira_vencida"] / df["carteira_ativa"], 0.0)
        df["fragilidadeRenda"] = np.where(df["carteira_ativa"] > 0, df["carteira_ativa_dh" ] / df["carteira_ativa"], 0.0)

        chave_uf = list(dict.fromkeys(chave + ["uf"]))
        pesos = ativa.groupby(chave_uf)["valor"].sum().rename("peso").reset_index()
        pesos = pesos.merge(escolarizacao_uf_mes(pesos), on=["ano_mes", "uf"], how="left").dropna(subset=["taxa_escolarizacao"])
        pesos["peso_x_taxa"] = pesos["peso"] * pesos["taxa_escolarizacao"]
        esc = pesos.groupby(chave).agg(peso=("peso", "sum"), peso_x_taxa=("peso_x_taxa", "sum")).reset_index()
        esc["taxa_escolarizacao"] = np.where(esc["peso"] > 0, esc["peso_x_taxa"] / esc["peso"], np.nan)

        df = df.merge(esc[chave + ["taxa_escolarizacao"]], on=chave, how="left")
        df["vulnerabilidadeSocial"] = 1 - df["taxa_escolarizacao"]

        for col in cols_ind:
            df = normalizacao(df, col)

        df[cols_ind] = df[cols_ind].replace([np.inf, -np.inf], np.nan).fillna(0)
        return df[chave + cols_ind]

    def plotar_painel(df, col_rotulo, titulo_fig, nome_eixo, horizontal=False, rotulo_media="Média"):
        painel = []
        for col, titulo, cor in indicadores:
            dados = df[[col_rotulo, col]].copy()
            dados[col] = dados[col].where(dados[col] > 0)
            painel.append((col, titulo, cor, dados.dropna().sort_values(col, ascending=False)))

        figsize = (16, max(10, 2 * (0.30 * max([len(p[3]) for p in painel] + [1]) + 1.5) + 1)) if horizontal else (16, 10)
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        fig.suptitle(titulo_fig, fontsize=16, fontweight="bold", color="#1a1a1a")

        for ax, (col, titulo, cor, dados) in zip(axes.flat, painel):
            ax.set_title(titulo, fontsize=12, fontweight="bold", pad=12)
            ax.set_xlabel("Escala 1 a 5" if horizontal else nome_eixo, fontsize=10)
            ax.set_ylabel(nome_eixo if horizontal else "Escala 1 a 5", fontsize=10)

            if dados.empty:
                ax.text(0.5, 0.5, "Sem dados no período", ha="center", va="center", transform=ax.transAxes, fontsize=12, color="gray")
                ax.set_xticks([])
                ax.set_yticks([])
                continue

            media = dados[col].mean()
            rotulo_leg = f"{rotulo_media} ({media:.2f})"
            if horizontal:
                sns.barplot(data=dados, y=col_rotulo, x=col, orient="h", color=cor, ax=ax, edgecolor="black", linewidth=0.5)
                ax.axvline(media, color="red", linestyle="--", linewidth=1.5, label=rotulo_leg)
                ax.set_xlim(0, 5.5)
                ax.tick_params(axis="y", labelsize=9)
                ax.grid(True, axis="x", linestyle="--", alpha=0.3)
                ax.legend(loc="lower right", frameon=True, fontsize=9)
            else:
                sns.barplot(data=dados, x=col_rotulo, y=col, color=cor, ax=ax, edgecolor="black", linewidth=0.5)
                ax.axhline(media, color="red", linestyle="--", linewidth=1.5, label=rotulo_leg)
                ax.set_ylim(0, 5.5)
                ax.grid(True, axis="y", linestyle="--", alpha=0.3)
                ax.legend(loc="upper right", frameon=True, fontsize=9)

            for container in ax.containers:
                ax.bar_label(container, fmt="%.2f", padding=3, fontsize=8 if horizontal else 9, fontweight="bold")

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

    # Verificação de simulação gráfica
    simulado = df_scr_pix is None or df_ibge is None

    if simulado:
        rng = np.random.default_rng(42)
        estrutura = {"Norte": 3, "Nordeste": 4, "Centro-Oeste": 3, "Sudeste": 4, "Sul": 3}
        linhas = []
        for regiao, n in estrutura.items():
            for i in range(1, n + 1):
                linhas.append({"ano_mes": "202412", "regiao": regiao, "rotulo": f"{regiao} - Sub {i}",
                               **{c: rng.uniform(1, 5) for c in cols_ind}})
        df_sub = pd.DataFrame(linhas)
        df_regiao = df_sub.groupby(["ano_mes", "regiao"], as_index=False)[cols_ind].mean()
    else:
        df_regiao = calcular_nivel(["regiao"])
        df_sub = calcular_nivel(["regiao", "uf", "sub_regiao"] if subregiao_por_uf else ["regiao", "sub_regiao"])

        if usar_nomes_subregiao:
            mapa = obter_mapeamento_subregioes()
            nome = df_sub["sub_regiao"].map(lambda x: f"{mapa[x]} ({x})" if x in mapa else f"Sub-região {x}")
        else:
            nome = df_sub["sub_regiao"]
        df_sub["rotulo"] = nome + " [" + df_sub["uf"] + "]" if subregiao_por_uf else nome

    # Filtro temporal
    mais_recente = df_regiao["ano_mes"].max()
    if periodo_ref is not None:
        periodo_ref = str(periodo_ref)
    else:
        preenchido = (df_regiao[cols_ind] > 0).groupby(df_regiao["ano_mes"]).any()
        completos = preenchido.index[preenchido.all(axis=1)]
        periodo_ref = completos.max() if len(completos) else mais_recente

    # Plotagem
    if mostrar_graficos:
        tag_sim = " | SIMULAÇÃO" if simulado else ""
        plotar_painel(
            df_regiao[df_regiao["ano_mes"] == periodo_ref],
            col_rotulo="regiao",
            titulo_fig=f"Eixo I – Indicadores de Risco de Crédito\n[Visão por REGIÃO | Período: {periodo_ref}{tag_sim}]",
            nome_eixo="Região",
            rotulo_media="Média das regiões",
        )

        for regiao_ref in sorted(df_regiao["regiao"].unique()):
            df_s = df_sub[(df_sub["ano_mes"] == periodo_ref) & (df_sub["regiao"] == regiao_ref)]
            if df_s.empty: continue
            plotar_painel(
                df_s,
                col_rotulo="rotulo",
                titulo_fig=f"Eixo I – Indicadores de Risco de Crédito\n[Região: {regiao_ref.upper()} | Sub-regiões | Período: {periodo_ref}{tag_sim}]",
                nome_eixo="Sub-região",
                horizontal=True,
                rotulo_media="Média da região",
            )

    return (None, None) if simulado else (df_regiao, df_sub)

df_eixo_i_regiao, df_eixo_i_sub_regiao = plotar_eixo_i_regioes_subregioes(
    globals().get("df_scr_pix"), globals().get("df_ibge"), periodo_ref=PERIODO
)



# **Eixo II – Inclusão e Expansão Demográfica**

---



In [ ]:
# --- Mudança feita dia 24-09 (Miguel) ---

def calcular_eixo_ii(df_scr_pix_base, df_ibge_base):
    df_scr_pix = df_scr_pix_base.copy()
    df_ibge = df_ibge_base.copy()

    # --- Extrai ano e merge com IBGE ---
    df_scr_pix["ano"] = df_scr_pix["ano_mes"].astype(str).str[:4]

    df_calculos = df_scr_pix.merge(
        df_ibge[["ano", "regiao", "uf", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]],
        on=["ano", "regiao", "uf"],
        how="left"
    ).drop(columns=["ano"])

    def eliminar_duplicata(metrica):
        df_calculos[metrica] = (
            df_calculos.groupby("uf")[metrica]
            .transform(lambda x: x.ffill().bfill())
        )

    eliminar_duplicata("populacao_residente")
    eliminar_duplicata("taxa_crescimento")
    eliminar_duplicata("variacao_populacao")
    eliminar_duplicata("bonusDemografico")

    # --- Agrupamento das métricas Pix ---
    df_metricas = pd.concat([
        agrupar_metrica(df_calculos, "vl_pagador").rename("vl_pagador"),
        agrupar_metrica(df_calculos, "qt_pes_pagador").rename("qt_pes_pagador"),
        agrupar_metrica(df_calculos, "qt_pagador").rename("qt_pagador"),
    ], axis=1).reset_index()

    df_calculos = (
        df_calculos[["ano_mes", "regiao", "uf", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]]
        .drop_duplicates()
        .merge(df_metricas, on=filtro, how="left")
    )
        # [NOVO] Remove meses que só existem no Pix (202011, 202012, 202609): sem SCR não há carteira
    # e esses meses apareceriam com zeros falsos
    # df_calculos = df_calculos.dropna(subset=["carteira_ativa"])

    # --- CORREÇÃO DO CÁLCULO E DIVISÃO POR ZERO (GJ00-33) ---
    # 1. Maturidade Uso PIX = qt_pagador / qt_pes_pagador
    maturidade_uso = np.where(
        df_calculos["qt_pes_pagador"] > 0,
        df_calculos["qt_pagador"] / df_calculos["qt_pes_pagador"],
        0.0
    )

    # 2. Maturidade Financeira PIX = vl_pagador / qt_pagador
    maturidade_financeira = np.where(
        df_calculos["qt_pagador"] > 0,
        df_calculos["vl_pagador"] / df_calculos["qt_pagador"],
        0.0
    )

    # 3. Consolidado Maturidade PIX = (Uso * 0.6) + (Financeira * 0.4)
    df_calculos["maturidadePix"] = (maturidade_uso * 0.6) + (maturidade_financeira * 0.4)

    # Demais indicadores do Eixo II
    df_calculos["crescimentoPopulacional"] = df_calculos["taxa_crescimento"]
    df_calculos["totalHabitantes"] = df_calculos["populacao_residente"]

    # --- Normalização ---
    df_calculos = normalizacao(df_calculos, "maturidadePix")
    df_calculos = normalizacao(df_calculos, "crescimentoPopulacional")
    df_calculos = normalizacao(df_calculos, "totalHabitantes")
    df_calculos = normalizacao(df_calculos, "bonusDemografico")

    # --- Limpeza final de segurança ---
    df_calculos = df_calculos.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_calculos[["ano_mes", "regiao", "uf", "maturidadePix", "crescimentoPopulacional", "totalHabitantes", "bonusDemografico"]]

---
#**EXECUÇÃO PRINCIPAL**
---

In [ ]:
# --- Extração e Pré-Processamento ---
processar()

construir_df_scr_pix()
validar_scr_contra_origem()   # [NOVO]
construir_df_ibge()

df_scr_pix = extract_data('/content/dados/scr_pix.parquet')
df_ibge = extract_data('/content/dados/ibge.parquet')

df_eixo_i = calcular_eixo_i(df_scr_pix, df_ibge)
df_eixo_ii = calcular_eixo_ii(df_scr_pix, df_ibge)

df_ibge.drop(columns=["bonusDemografico"], inplace=True)

# --- Limpeza de segurança contra NaNs e infinitos antes do envio ---
df_eixo_i = df_eixo_i.replace([np.inf, -np.inf], np.nan).fillna(0)
df_eixo_ii = df_eixo_ii.replace([np.inf, -np.inf], np.nan).fillna(0)
# [NOVO] "classe" é uma coluna de categorias e fica vazia nas linhas do Pix.
# O fillna(0) não consegue colocar o 0 numa categoria que não existe, então elas viram texto antes
colunas_categoricas = df_scr_pix.select_dtypes("category").columns
df_scr_pix[colunas_categoricas] = df_scr_pix[colunas_categoricas].astype(object)

df_scr_pix = df_scr_pix.replace([np.inf, -np.inf], np.nan).fillna(0)   # [IGUAL]
df_ibge = df_ibge.replace([np.inf, -np.inf], np.nan).fillna(0)

---
# **Visualização Gráfica – Eixo II: Inclusão e Expansão Demográfica**
---
### Critérios de Aceite Atendidos:
- **Gráficos dos 4 indicadores:** Maturidade Pix, Crescimento Populacional, Total de Habitantes e Bônus Demográfico.
- **Filtros interativos:** Possibilidade de filtrar por período (Ano/Mês) e região (Macrorregião ou Brasil).
- **Formatação completa:** Cada gráfico contém título, eixos nomeados (X e Y) e legenda explicativa.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def visualizar_indicadores_eixo_ii(df_dados, periodo="Todos", regiao="Todas", salvar_caminho=None):
    """
    Gera os 4 gráficos do Eixo II - Inclusão e Demografia com filtros de período e região.
    Atende integralmente aos critérios de aceite:
    1. Gráficos de maturidade Pix, crescimento populacional, total de habitantes e bônus demográfico
    2. Filtros por período e região
    3. Título, eixos nomeados e legenda
    """
    df_f = df_dados.copy()
    
    # 1. Filtro por Período
    if periodo and str(periodo).lower() not in ["todos", "todas", "all"]:
        periodo_str = str(periodo)
        if len(periodo_str) == 4:
            df_f = df_f[df_f["ano_mes"].astype(str).str.startswith(periodo_str)]
        else:
            df_f = df_f[df_f["ano_mes"].astype(str) == periodo_str]
            
    # 2. Filtro por Região
    if regiao and str(regiao).lower() not in ["todas", "todos", "brasil", "todas as regiões", "all"]:
        regiao_str = str(regiao).strip()
        if regiao_str in df_f["regiao"].values:
            df_f = df_f[df_f["regiao"] == regiao_str]
        elif regiao_str in df_f["uf"].values:
            df_f = df_f[df_f["uf"] == regiao_str]
            
    if df_f.empty:
        print(f"Nenhum registro para Período='{periodo}' e Região='{regiao}'.")
        return None
        
    df_agg = df_f.groupby(["uf", "regiao"], as_index=False).agg({
        "maturidadePix": "mean",
        "crescimentoPopulacional": "mean",
        "totalHabitantes": "mean",
        "bonusDemografico": "mean"
    }).sort_values("uf")
    
    ufs = df_agg["uf"].tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    fig.patch.set_facecolor("#f8fafc")
    
    plt.suptitle(
        f"Eixo II – Indicadores de Inclusão e Expansão Demográfica\n[Período: {periodo} | Região: {regiao}]",
        fontsize=16, fontweight="bold", color="#0f172a", y=0.98
    )
    
    # 1. Maturidade Pix
    ax1 = axes[0, 0]
    ax1.set_facecolor("#ffffff")
    b1 = ax1.bar(ufs, df_agg["maturidadePix"], color="#0284c7", alpha=0.85, edgecolor="#0369a1", label="Maturidade Pix (Normalizado 1-5)")
    ax1.axhline(df_agg["maturidadePix"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['maturidadePix'].mean():.2f})")
    ax1.set_title("1. Maturidade Pix por Estado", pad=10, fontweight="bold")
    ax1.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax1.set_ylabel("Índice de Maturidade Pix (Escala 1 a 5)", fontweight="bold")
    ax1.set_ylim(0, 5.5)
    ax1.legend(loc="upper right", frameon=True)
    ax1.grid(True, linestyle=":", alpha=0.6)
    for b in b1:
        h = b.get_height()
        ax1.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    # 2. Crescimento Populacional
    ax2 = axes[0, 1]
    ax2.set_facecolor("#ffffff")
    b2 = ax2.bar(ufs, df_agg["crescimentoPopulacional"], color="#16a34a", alpha=0.85, edgecolor="#15803d", label="Crescimento Populacional (Normalizado 1-5)")
    ax2.axhline(df_agg["crescimentoPopulacional"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['crescimentoPopulacional'].mean():.2f})")
    ax2.set_title("2. Crescimento Populacional por Estado (Censo IBGE)", pad=10, fontweight="bold")
    ax2.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax2.set_ylabel("Índice de Crescimento Populacional (Escala 1 a 5)", fontweight="bold")
    ax2.set_ylim(0, 5.5)
    ax2.legend(loc="upper right", frameon=True)
    ax2.grid(True, linestyle=":", alpha=0.6)
    for b in b2:
        h = b.get_height()
        ax2.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    # 3. Total de Habitantes
    ax3 = axes[1, 0]
    ax3.set_facecolor("#ffffff")
    b3 = ax3.bar(ufs, df_agg["totalHabitantes"], color="#8b5cf6", alpha=0.85, edgecolor="#6d28d9", label="Total de Habitantes (Normalizado 1-5)")
    ax3.axhline(df_agg["totalHabitantes"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['totalHabitantes'].mean():.2f})")
    ax3.set_title("3. Total de Habitantes por Estado (População Residente)", pad=10, fontweight="bold")
    ax3.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax3.set_ylabel("Índice de Total de Habitantes (Escala 1 a 5)", fontweight="bold")
    ax3.set_ylim(0, 5.5)
    ax3.legend(loc="upper right", frameon=True)
    ax3.grid(True, linestyle=":", alpha=0.6)
    for b in b3:
        h = b.get_height()
        ax3.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    # 4. Bônus Demográfico
    ax4 = axes[1, 1]
    ax4.set_facecolor("#ffffff")
    b4 = ax4.bar(ufs, df_agg["bonusDemografico"], color="#f59e0b", alpha=0.85, edgecolor="#d97706", label="Bônus Demográfico (Normalizado 1-5)")
    ax4.axhline(df_agg["bonusDemografico"].mean(), color="#dc2626", linestyle="--", linewidth=1.5, label=f"Média ({df_agg['bonusDemografico'].mean():.2f})")
    ax4.set_title("4. Bônus Demográfico por Estado (Janela Etária)", pad=10, fontweight="bold")
    ax4.set_xlabel("Unidade Federativa (UF)", fontweight="bold")
    ax4.set_ylabel("Índice de Bônus Demográfico (Escala 1 a 5)", fontweight="bold")
    ax4.set_ylim(0, 5.5)
    ax4.legend(loc="upper right", frameon=True)
    ax4.grid(True, linestyle=":", alpha=0.6)
    for b in b4:
        h = b.get_height()
        ax4.annotate(f"{h:.2f}", (b.get_x() + b.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")
        
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    if salvar_caminho:
        plt.savefig(salvar_caminho, dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Filtros Interativos por Período e Região para o Google Colab
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    # Obter opções disponíveis a partir do df_eixo_ii
    periodos_disponiveis = ["Todos"] + sorted(df_eixo_ii["ano_mes"].astype(str).unique().tolist())
    regioes_disponiveis = ["Todas"] + sorted(df_eixo_ii["regiao"].dropna().unique().tolist())
    
    dropdown_periodo = widgets.Dropdown(
        options=periodos_disponiveis,
        value="Todos",
        description="Período:",
        style={"description_width": "initial"}
    )
    
    dropdown_regiao = widgets.Dropdown(
        options=regioes_disponiveis,
        value="Todas",
        description="Região:",
        style={"description_width": "initial"}
    )
    
    def atualizar(periodo, regiao):
        visualizar_indicadores_eixo_ii(df_eixo_ii, periodo=periodo, regiao=regiao)
        
    display(widgets.interactive(atualizar, periodo=dropdown_periodo, regiao=dropdown_regiao))
except Exception as e:
    print(f"Executando visualização estática (ipywidgets opcional): {e}")
    visualizar_indicadores_eixo_ii(df_eixo_ii, periodo="Todos", regiao="Todas")


In [ ]:
# Exemplo: Visualização dos gráficos filtrando por Região="Nordeste" e Período mais recente
visualizar_indicadores_eixo_ii(df_eixo_ii, periodo="Todos", regiao="Nordeste")
